In [15]:
import pandas as pd
import numpy as np
import json
import sys

sys.path.append('/home/mdafifal.mamun/research/S3M')

In [16]:
prediction_path = "/home/mdafifal.mamun/research/S3M/saved_predictions/eclipse_dedupt_predictions_1771135140.5734932.json"

In [17]:
with open(prediction_path, 'r') as f:
    predictions = json.load(f)

In [18]:
dedupt_processed_results = []

for _, result in predictions.items():
    bug_id = result["bug_id"]
    actual_duplicate_id = result["actual_duplicate_id"]
    preds = [int(pred) for pred in result["predictions"].keys()]

    dedupt_processed_results.append((bug_id, actual_duplicate_id, preds))
    

In [19]:
rows = []

for dedupt_result in dedupt_processed_results:
    bug_id, actual_duplicate_id, dedupt_preds = dedupt_result
    
    rows.append({
        "bug_id": bug_id,
        "actual_duplicate_id": actual_duplicate_id,
        "dedupt_preds": dedupt_preds[:10],        
    })
result_df = pd.DataFrame(rows)

def reciprocal_rank(preds, actual):
    try:
        rank = preds.index(actual) + 1
        return 1.0 / rank
    except ValueError:
        return 0.0


result_df["rr_dedupt"] = result_df.apply(
    lambda x: reciprocal_rank(x["dedupt_preds"], x["actual_duplicate_id"]),
    axis= 1
)

## Prediction Analysis

In [32]:
analytical_samples = result_df.sample(n=100, random_state=500)
analytical_samples["rr_dedupt"].mean()

0.8110119047619048

In [44]:
analytical_samples[analytical_samples["bug_id"] == 461384]

,bug_id,actual_duplicate_id,dedupt_preds,rr_dedupt
1915,461384,441721,"[461119, 461018, 445579, 441721, 453149, 45280...",0.25


In [37]:
analytical_samples

,bug_id,actual_duplicate_id,dedupt_preds,rr_dedupt
299,444552,443906,"[443906, 433552, 417764, 223381, 281455, 43933...",1.00
504,446468,445457,"[445457, 445983, 444102, 445116, 445782, 44400...",1.00
1915,461384,441721,"[461119, 461018, 445579, 441721, 453149, 45280...",0.25
695,447164,394512,"[394512, 439159, 442783, 358491, 441692, 43406...",1.00
1979,464750,441721,"[441721, 461119, 462048, 461018, 445579, 45314...",1.00
...,...,...,...,...
524,446567,445459,"[445459, 442806, 421812, 344861, 445185, 43461...",1.00
653,447015,394512,"[394512, 439159, 358491, 442783, 441692, 43328...",1.00
312,444689,436896,"[436896, 427064, 441744, 425280, 424749, 42477...",1.00
2072,468921,460618,"[460618, 454143, 443312, 450875, 458870, 45092...",1.00


In [39]:
analytical_samples.to_csv("analytical_samples.csv", index=False)

In [38]:
analytical_samples["rr_dedupt"].value_counts()

rr_dedupt
1.000000    75
0.000000     9
0.500000     9
0.250000     4
0.125000     1
0.142857     1
0.333333     1
Name: count, dtype: int64

## Effort Analysis

In [23]:
full_dataset = "/home/mdafifal.mamun/research/S3M/dataset/EMSE_data/eclipse_2018/eclipse_stacktraces.json"

with open(full_dataset, 'r') as f:
    full_data = json.load(f)

In [24]:
full_data_df = pd.DataFrame(full_data)

In [25]:
num_frames = 10
def format_stack(stack, from_top=False):
    # Remove duplicate frames
    exception = stack.get("exception", "UnknownException")
    message = stack.get("message", "")
    frames = stack.get("frames", [])
    frames = [f"{f['function']} at {f['file']}:{f['fileline']}" for f in frames]            

    frames = "\n".join([f"{i+1}: {frame}" for i, frame in enumerate(frames)][:num_frames])
    return f"{exception}: {message}\n{frames}"


def get_bug_repot(bug_id):
    bug_report = full_data_df[full_data_df["bug_id"] == bug_id]
    br = f"""
    Bug ID: {bug_report.iloc[0]['bug_id']}
    Title: {bug_report.iloc[0]['short_desc']}

{format_stack(bug_report.iloc[0]['stacktrace'][0])}
"""
    return br

In [46]:
def check_predictions(analytical_sample):
    bug_id = analytical_sample["bug_id"]
    preds = analytical_sample["dedupt_preds"]
    print(f"Actual Bug ID: {bug_id}")
    print(get_bug_repot(bug_id))
    print("*" * 50)
    print("Predicted Duplicates:")
    for pred in preds[:5]:
        print(get_bug_repot(pred))
        print("-" * 50)
    print("=" * 50)


for _, sample in analytical_samples.head(20).iterrows():
    print(f"Analyzing Bug ID: {sample['bug_id']} with Actual Duplicate ID: {int(sample['actual_duplicate_id'])}")
    check_predictions(sample)

Analyzing Bug ID: 444552 with Actual Duplicate ID: 443906
Actual Bug ID: 444552

    Bug ID: 444552
    Title: Internal Error (err_grp: 753aa95b)

org.eclipse.core.internal.resources.ResourceException: Resource '/com.codetrails.ctrlflow.codesearch.rcp/src-gen/com/codetrails/ctrlflow/codesearch/rcp/model/ArgStatmentSnippet.java' does not exist.
1: org.eclipse.core.internal.resources.Resource.checkExists at Resource.java:341
2: org.eclipse.core.internal.resources.Resource.checkAccessible at Resource.java:215
3: org.eclipse.core.internal.resources.Resource.findMaxProblemSeverity at Resource.java:1051
4: org.eclipse.jdt.ui.ProblemsLabelDecorator.getPackageErrorTicksFromMarkers at ProblemsLabelDecorator.java:337
5: org.eclipse.jdt.ui.ProblemsLabelDecorator.computeAdornmentFlags at ProblemsLabelDecorator.java:212
6: org.eclipse.jdt.internal.ui.viewsupport.TreeHierarchyLayoutProblemsDecorator.computePackageAdornmentFlags at TreeHierarchyLayoutProblemsDecorator.java:47
7: org.eclipse.jdt.inter

## Confidence Interval

In [47]:
from scipy import stats

df = analytical_samples
n = len(df)
rr = df['rr_dedupt'].values

# Core stats
mean_rr = np.mean(rr)
std_rr = np.std(rr, ddof=1)
se = stats.sem(rr)
ci = stats.t.interval(0.95, df=n-1, loc=mean_rr, scale=se)
margin = (ci[1] - ci[0]) / 2


# RR@k
rr1 = (rr == 1.0).sum()
rr5 = (rr >= 0.2).sum()  # rank <= 5 means RR >= 1/5 = 0.2
rr10 = (rr > 0.0).sum()  # found within top 10

# Rank distribution
rank_dist = {}
for v in rr:
    if v == 0:
        rank_dist['Not found'] = rank_dist.get('Not found', 0) + 1
    else:
        rank = int(round(1/v))
        rank_dist[f'Rank {rank}'] = rank_dist.get(f'Rank {rank}', 0) + 1

print(f"N: {n}")
print(f"Mean MRR: {mean_rr:.4f}")
print(f"95% CI: ({ci[0]:.4f}, {ci[1]:.4f})")
print(f"Margin of Error: ±{margin:.4f}")
print(f"\nRR@1 (rank-1 correct): {rr1}/{n} = {rr1/n:.4f}")
print(f"RR@5 (found within top-5): {rr5}/{n} = {rr5/n:.4f}")
print(f"RR@10 (found within top-10): {rr10}/{n} = {rr10/n:.4f}")
print(f"Not found: {n - rr10}/{n}")
print(f"\nRank distribution: {dict(sorted(rank_dist.items()))}")


N: 100
Mean MRR: 0.8110
95% CI: (0.7423, 0.8797)
Margin of Error: ±0.0687

RR@1 (rank-1 correct): 75/100 = 0.7500
RR@5 (found within top-5): 89/100 = 0.8900
RR@10 (found within top-10): 91/100 = 0.9100
Not found: 9/100

Rank distribution: {'Not found': 9, 'Rank 1': 75, 'Rank 2': 9, 'Rank 3': 1, 'Rank 4': 4, 'Rank 7': 1, 'Rank 8': 1}
